# 01 拉取市场数据（交易日历 + 指数日线）

目的：拉取交易日历与宽基指数日线数据，缓存到 `data/cache/`，供后续“流日干支 → A股涨跌”研究使用。

输出：
- `data/cache/trade_cal/trade_cal.csv.gz`
- `data/cache/index_daily/{ts_code}.csv.gz`

安全：Token 仅从环境变量 `TUSHARE_API_KEY` 或 `.tushare_token/~/.tushare_token` 读取；不会打印完整 Token。


In [1]:
from __future__ import annotations

import datetime as _dt
import os
from pathlib import Path

import pandas as pd
import tushare as ts

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    # Typical: notebook is executed under project root or under ./notebooks
    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    # Fallback: if executed from a parent folder, look for a child project dir
    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)


# =====================
# 配置区（按需修改）
# =====================
START_DATE = '20100101'
END_DATE = _dt.date.today().strftime('%Y%m%d')
FORCE_REFRESH = False  # True: 忽略本地缓存，重新拉取

INDEX_LIST = {
    'HS300': '000300.SH',
    'CSI1000': '000852.SH',
    # 可选对照（接口不一定都可用；失败会跳过）
    'SSE': '000001.SH',
    'SZSE': '399001.SZ',
}

CACHE_DIR = ROOT / 'data/cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def load_tushare_token() -> str:
    env_token = os.getenv('TUSHARE_API_KEY')
    if env_token and env_token.strip():
        return env_token.strip()

    for candidate in [ROOT / '.tushare_token', Path.home() / '.tushare_token']:
        if candidate.is_file():
            token = candidate.read_text(encoding='utf-8').strip()
            if token:
                return token

    raise RuntimeError(
        'No Tushare token found. Set env TUSHARE_API_KEY or put token in .tushare_token / ~/.tushare_token'
    )


def build_pro_client() -> object:
    token = load_tushare_token()
    pro = ts.pro_api(token)

    # 可选：通过环境变量覆盖 base url（与 tushare_api_docs/TUSHARE_CONFIG.md 对齐）
    base_url = os.getenv('TUSHARE_BASE_URL')
    if base_url:
        pro._DataApi__http_url = base_url.rstrip('/')  # type: ignore[attr-defined]
    return pro


def read_csv_gz(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, compression='gzip', dtype={'trade_date': str, 'cal_date': str})


def write_csv_gz(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, compression='gzip')


pro = build_pro_client()
print('END_DATE =', END_DATE)

END_DATE = 20260213


In [2]:
# 1) trade_cal（交易日历）
trade_cal_path = CACHE_DIR / 'trade_cal' / 'trade_cal.csv.gz'

if trade_cal_path.exists() and not FORCE_REFRESH:
    trade_cal = read_csv_gz(trade_cal_path)
    print('Loaded cache:', trade_cal_path)
else:
    trade_cal = pro.trade_cal(
        exchange='',
        start_date=START_DATE,
        end_date=END_DATE,
        fields='exchange,cal_date,is_open,pretrade_date',
    )
    trade_cal = trade_cal.sort_values(['cal_date']).reset_index(drop=True)
    write_csv_gz(trade_cal, trade_cal_path)
    print('Saved cache:', trade_cal_path)

print('trade_cal rows =', len(trade_cal))
print('open days =', int(trade_cal['is_open'].sum()))
print('range =', trade_cal['cal_date'].min(), '->', trade_cal['cal_date'].max())


Saved cache: data\cache\trade_cal\trade_cal.csv.gz
trade_cal rows = 5888
open days = 3916
range = 20100101 -> 20260213


In [3]:
# 2) index_daily（指数日线）
idx_dir = CACHE_DIR / 'index_daily'
idx_dir.mkdir(parents=True, exist_ok=True)

summary = []
for name, ts_code in INDEX_LIST.items():
    out_path = idx_dir / f'{ts_code}.csv.gz'

    if out_path.exists() and not FORCE_REFRESH:
        df = read_csv_gz(out_path)
        print(f'[{name}] Loaded cache:', out_path)
    else:
        try:
            df = pro.index_daily(
                ts_code=ts_code,
                start_date=START_DATE,
                end_date=END_DATE,
                fields='ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount',
            )
        except Exception as exc:
            print(f'[{name}] Fetch failed for {ts_code}: {exc}')
            continue

        if df.empty:
            print(f'[{name}] Empty result for {ts_code} (skip)')
            continue

        df = df.sort_values(['trade_date']).reset_index(drop=True)
        write_csv_gz(df, out_path)
        print(f'[{name}] Saved cache:', out_path)

    df['trade_date'] = df['trade_date'].astype(str)
    summary.append(
        {
            'name': name,
            'ts_code': ts_code,
            'rows': len(df),
            'min_trade_date': df['trade_date'].min(),
            'max_trade_date': df['trade_date'].max(),
        }
    )

pd.DataFrame(summary)


[HS300] Saved cache: data\cache\index_daily\000300.SH.csv.gz
[CSI1000] Saved cache: data\cache\index_daily\000852.SH.csv.gz
[SSE] Saved cache: data\cache\index_daily\000001.SH.csv.gz
[SZSE] Saved cache: data\cache\index_daily\399001.SZ.csv.gz


,name,ts_code,rows,min_trade_date,max_trade_date
0,HS300,000300.SH,3916,20100104,20260213
1,CSI1000,000852.SH,3916,20100104,20260213
2,SSE,000001.SH,3916,20100104,20260213
3,SZSE,399001.SZ,3916,20100104,20260213
